<a href="https://colab.research.google.com/github/JulietChinenyeDuru/KG-SentIF-Knowledge-Graph-Enhanced-Sentiment-Analysis-for-Nigerian-Social-Media-Text/blob/main/notebooks/google_play_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NaijaABSA: collect Google Play reviews

Run each cell from top to bottom (click ▶ or press **Shift + Enter**). No GPU is needed.

**What this notebook does**
1. Uses a fixed list of nine Nigerian apps
2. Downloads recent reviews (reviewer names are never saved)
3. Removes personal details, duplicates and very short or long reviews
4. Keeps reviews containing Nigerian Pidgin markers
5. Samples up to 300 per domain and downloads the files to your computer

## 1. Install the library

In [ ]:
!pip install -q google-play-scraper

## 2. The apps

Nine Nigerian apps across four domains. To add an app, open its Google Play page and copy the part after `id=` in the web address.

In [ ]:
APPS = {
    "telecoms":    ["ng.mtn.nextgen",                         # myMTN NG
                    "net.one97.selfcare.globacom"],           # Glo Cafe Nigeria
    "banking":     ["com.firstbank.firstmobile",              # FirstMobile (First Bank)
                    "com.moniepoint.business"],               # Moniepoint Business
    "electricity": ["com.powerlight.ng",                      # Powerlight
                    "com.ionicframework.buypowerapp334891",   # BuyPower
                    "com.vatebrapay.fcmbenergy"],             # Ikeja Electric
    "ecommerce":   ["com.jumia.android",                      # Jumia
                    "ng.jiji.app"],                           # Jiji Nigeria
}
MAX_PAGES = 50   # 50 x 200 = up to 10,000 reviews per app

## 3. Download the reviews
With `MAX_PAGES = 50`

In [ ]:
import time
import pandas as pd
from google_play_scraper import reviews, Sort

rows = []
for domain, app_ids in APPS.items():
    for app_id in app_ids:
        token, n = None, 0
        for _ in range(MAX_PAGES):
            try:
                batch, token = reviews(app_id, lang="en", country="ng", sort=Sort.NEWEST,
                                       count=200, continuation_token=token)
            except Exception as e:
                print("  stopped", app_id, "-", e); break
            for r in batch:                         # reviewer name is NOT saved
                rows.append({"review_id": r["reviewId"], "app_id": app_id, "domain": domain,
                             "text": r["content"], "stars": r["score"], "date": str(r["at"])})
            n += len(batch)
            if not batch or token is None:
                break
            time.sleep(2)                           # be polite to the server
        print(f"{domain:12s} {app_id:45s} {n} reviews")

raw = pd.DataFrame(rows)
raw.to_json("playstore_raw.jsonl", orient="records", lines=True)
print("Total:", len(raw))

## 4. Clean, filter for Pidgin and sample

In [ ]:
import re

df = raw.copy()

def scrub(t):
    t = re.sub(r"\S+@\S+", "[EMAIL]", str(t))
    t = re.sub(r"(\+?234|0)[789][01]\d{8}", "[PHONE]", t)
    t = re.sub(r"@\w+", "[USER]", t)
    t = re.sub(r"https?://\S+", "[URL]", t)
    return t.strip()

df["text"] = df["text"].map(scrub)
df = df.drop_duplicates("text")
df["n_words"] = df["text"].str.split().str.len()
df = df[(df.n_words >= 5) & (df.n_words <= 80)]

MARKERS = ["dey", "wey", "abeg", "no be", "una", "wetin", "sabi", "oga", "wahala",
           "comot", "abi", "sha", "shey", "e don", "don dey", "na im", "nawa", "chop",
           "o jare", "biko", "dem", "wan", "make una", "e no"]
pattern = r"\b(" + "|".join(re.escape(m) for m in MARKERS) + r")\b"
df["pidgin_hits"] = df["text"].str.lower().str.count(pattern)

print("After cleaning:", len(df))
print("With at least 1 Pidgin marker:", (df.pidgin_hits >= 1).sum())
print("With at least 2 Pidgin markers:", (df.pidgin_hits >= 2).sum())
print()
print(df[df.pidgin_hits >= 1].groupby("domain").size())

### Check by eye
Read the sample below. If many are ordinary English, change `MIN_HITS` to 2 in the next cell.

In [ ]:
pd.set_option("display.max_colwidth", 200)
df[df.pidgin_hits >= 1].sample(min(30, (df.pidgin_hits >= 1).sum()), random_state=1)[["domain", "text"]]

In [ ]:
MIN_HITS = 1
PER_DOMAIN = 300

pidgin = df[df.pidgin_hits >= MIN_HITS]
sample = pd.concat([g.sample(min(len(g), PER_DOMAIN), random_state=42)
                    for _, g in pidgin.groupby("domain")])
sample[["review_id", "app_id", "domain", "text"]].to_json(
    "to_annotate.jsonl", orient="records", lines=True)
sample[["review_id", "app_id", "domain", "text"]].to_csv("to_annotate.csv", index=False)
print(sample.domain.value_counts())

## 5. Download the files

- `to_annotate.jsonl` – import this into Label Studio (Step 3 of the guide)
- `to_annotate.csv` – the same posts, easy to open in Excel
- `playstore_raw.jsonl` – everything collected; keep it private and **do not** upload it to GitHub



In [ ]:
# I upload my review file and count the reviews in each domain

import io
import pandas as pd
from google.colab import files

uploaded = files.upload()
name = list(uploaded.keys())[0]
df = pd.read_json(io.BytesIO(uploaded[name]), lines=True)

print("Reading:", name)
print(len(df), "reviews")
print(df.domain.value_counts())
df.head(20)

Saving to_annotate.jsonl to to_annotate (1).jsonl
Reading: to_annotate (1).jsonl
435 reviews
domain
telecoms       300
banking         65
ecommerce       42
electricity     28
Name: count, dtype: int64


,review_id,app_id,domain,text
0,a6579562-fc33-4615-8b9a-2162159ab419,com.moniepoint.business,banking,no be we they benefit and the agent and the co...
1,4475f1a8-2220-4c9c-8e3c-19043144624a,com.moniepoint.business,banking,He dey care for customers well well
2,c4e544d8-68eb-47bd-9919-7a94aa5fb00a,com.firstbank.firstmobile,banking,make una dey try give me loans
3,0598dd07-4fda-4d11-89c0-dade693f2094,com.moniepoint.business,banking,Moniepoint dey traumatize me gan
4,3dd4d226-dccb-4db4-b269-eeb0e23ef897,com.firstbank.firstmobile,banking,why did u pple start ledger balance again. u s...
5,c2f15290-ed5c-46f0-b765-8f72b3706115,com.moniepoint.business,banking,the fact that you don't use to have network pr...
6,680253bb-a54e-46de-966c-26e944e4ef53,com.firstbank.firstmobile,banking,I don't know why the app has lodge me out it k...
7,628fe683-2018-4767-a2a7-5710b9eb120c,com.firstbank.firstmobile,banking,"Please work on your newly updated app, login i..."
8,7d8105f6-550e-4c78-94db-74462f72beaa,com.moniepoint.business,banking,I like as everybody dey throw deep quote aroun...
9,85034a94-905c-4370-a435-4305b8303ee2,com.firstbank.firstmobile,banking,My experience with this app is very horrible a...


In [ ]:
# I build my two pilot annotation sheets from my review file and download them

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
import glob
import pandas as pd
from google.colab import files

CATEGORIES = ["product overall", "price", "network or coverage", "data and airtime",
              "customer service", "app performance", "login and security",
              "transfers and payments", "fees and charges", "power supply",
              "billing and metering", "token and recharge", "delivery",
              "product quality", "seller trust", "refunds"]
POLARITY = ["positive", "negative", "neutral"]
YESNO = ["yes", "no"]
LANGUAGE = ["English", "Pidgin", "Code-switched"]

F = "Arial"
HEAD = PatternFill("solid", fgColor="1F4E79")
INPUT = PatternFill("solid", fgColor="FFF9DB")
EXAMPLE = PatternFill("solid", fgColor="E2EFDA")
thin = Side(style="thin", color="BFBFBF")
BOX = Border(left=thin, right=thin, top=thin, bottom=thin)

COLS = [("ID", 12), ("Domain", 13), ("Review", 60),
        ("Aspect 1 (exact words)", 20), ("Category 1", 22), ("Polarity 1", 12),
        ("Aspect 2 (exact words)", 20), ("Category 2", 22), ("Polarity 2", 12),
        ("Aspect 3 (exact words)", 20), ("Category 3", 22), ("Polarity 3", 12),
        ("Implied aspect category", 22), ("Implied aspect polarity", 14),
        ("Sarcastic?", 11), ("Cultural reference?", 13), ("Language", 15), ("Notes", 30)]

EXAMPLES = [
    ["EXAMPLE-1", "banking", "This app sweet well well, but the transfer no dey go at all",
     "app", "product overall", "positive", "transfer", "transfers and payments", "negative",
     "", "", "", "", "", "no", "no", "Pidgin", "Two aspects, opposite feelings"],
    ["EXAMPLE-2", "telecoms", "Thank you MTN for the data wey finish in one hour",
     "data", "data and airtime", "negative", "", "", "", "", "", "", "", "", "yes", "no",
     "Pidgin", "Sarcasm: polite words, negative meaning"],
    ["EXAMPLE-3", "electricity", "I don buy token since morning, nothing show for my meter",
     "token", "token and recharge", "negative", "", "", "", "", "", "", "", "", "no", "no",
     "Pidgin", ""],
    ["EXAMPLE-4", "electricity", "Na so una dey do every time, abeg do better",
     "", "", "", "", "", "", "", "", "", "power supply", "negative", "no", "yes",
     "Pidgin", "No aspect named, so use the implied columns"],
]

def make_sheet(df, path, text_col="text", id_col="review_id", domain_col="domain"):
    wb = Workbook()

    ins = wb.active
    ins.title = "Instructions"
    lines = [
        ("NaijaABSA annotation sheet", True),
        ("", False),
        ("What to do", True),
        ("1. Go to the 'Annotate' tab. Read each review in column C.", False),
        ("2. For each thing the writer gives an opinion about, type the EXACT words from the review in an Aspect column (e.g. transfer, data, delivery).", False),
        ("3. Choose its Category and Polarity from the dropdown menus. Up to 3 aspects per review.", False),
        ("4. If the opinion is about something never named in words, leave Aspect empty and fill the two 'Implied aspect' columns.", False),
        ("5. Choose yes/no for Sarcastic? and Cultural reference? (you need local knowledge to understand it), and choose the Language.", False),
        ("6. If a review has no opinion about a product or service, write 'no opinion' in Notes and leave the rest empty.", False),
        ("7. Replace any personal name you see with [NAME] in Notes; do not edit the review text.", False),
        ("", False),
        ("Colours", True),
        ("Yellow cells = fill these in. Green rows = worked examples (do not change them).", False),
        ("Work on your own. Do not discuss reviews with the other annotator.", False),
        ("", False),
        ("Categories", True),
    ] + [(f"- {c}", False) for c in CATEGORIES]
    for i, (t, bold) in enumerate(lines, 1):
        c = ins.cell(row=i, column=1, value=t)
        c.font = Font(name=F, bold=bold, size=13 if i == 1 else 11)
        c.alignment = Alignment(wrap_text=True, vertical="top")
    ins.column_dimensions["A"].width = 110

    ls = wb.create_sheet("Lists")
    for j, (name, vals) in enumerate([("Category", CATEGORIES), ("Polarity", POLARITY),
                                      ("YesNo", YESNO), ("Language", LANGUAGE)], 1):
        ls.cell(row=1, column=j, value=name).font = Font(name=F, bold=True)
        for i, v in enumerate(vals, 2):
            ls.cell(row=i, column=j, value=v).font = Font(name=F)
    ls.sheet_state = "hidden"

    ws = wb.create_sheet("Annotate", 1)
    for j, (h, w) in enumerate(COLS, 1):
        c = ws.cell(row=1, column=j, value=h)
        c.font = Font(name=F, bold=True, color="FFFFFF")
        c.fill = HEAD
        c.alignment = Alignment(wrap_text=True, vertical="center", horizontal="center")
        c.border = BOX
        ws.column_dimensions[c.column_letter].width = w
    ws.row_dimensions[1].height = 36
    ws.freeze_panes = "D2"

    rows = EXAMPLES + [[str(r[id_col]), r[domain_col], r[text_col]] + [""] * 15
                       for _, r in df.iterrows()]
    n = len(rows) + 1
    for i, row in enumerate(rows, 2):
        ex = i <= len(EXAMPLES) + 1
        for j, v in enumerate(row, 1):
            c = ws.cell(row=i, column=j, value=v if v != "" else None)
            c.font = Font(name=F, italic=ex)
            c.alignment = Alignment(wrap_text=(j == 3), vertical="top")
            c.border = BOX
            if ex:
                c.fill = EXAMPLE
            elif j >= 4:
                c.fill = INPUT

    def dv(formula, cols):
        v = DataValidation(type="list", formula1=formula, allow_blank=True,
                           showErrorMessage=True, errorTitle="Use the list",
                           error="Please choose a value from the dropdown.")
        ws.add_data_validation(v)
        for col in cols:
            v.add(f"{col}2:{col}{n}")
    dv(f"=Lists!$A$2:$A${len(CATEGORIES)+1}", ["E", "H", "K", "M"])
    dv("=Lists!$B$2:$B$4", ["F", "I", "L", "N"])
    dv("=Lists!$C$2:$C$3", ["O", "P"])
    dv("=Lists!$D$2:$D$4", ["Q"])

    wb.active = 1
    wb.save(path)
    return path

path = sorted(glob.glob("to_annotate*.jsonl"))[-1]
df = pd.read_json(path, lines=True)
pilot = df.sample(min(100, len(df)), random_state=1)

make_sheet(pilot, "pilot_annotator_A.xlsx")
make_sheet(pilot, "pilot_annotator_B.xlsx")
pilot.to_csv("pilot_review_ids.csv", index=False)
print("Pilot sheets created with", len(pilot), "reviews from", path)

files.download("pilot_annotator_A.xlsx")
files.download("pilot_annotator_B.xlsx")
files.download("pilot_review_ids.csv")

Pilot sheets created with 100 reviews from to_annotate.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# I create two balanced pilot sheets with 25 reviews from each domain

import glob
import pandas as pd
from google.colab import files

path = sorted(glob.glob("to_annotate*.jsonl"))[-1]
df = pd.read_json(path, lines=True)
pilot = pd.concat([g.sample(min(25, len(g)), random_state=1) for _, g in df.groupby("domain")])
pilot = pilot.sample(frac=1, random_state=2)

make_sheet(pilot, "pilot_annotator_A.xlsx")
make_sheet(pilot, "pilot_annotator_B.xlsx")
pilot.to_csv("pilot_review_ids.csv", index=False)
print(pilot.domain.value_counts())

files.download("pilot_annotator_A.xlsx")
files.download("pilot_annotator_B.xlsx")
files.download("pilot_review_ids.csv")

domain
telecoms       25
ecommerce      25
electricity    25
banking        25
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>